### nxsut generator — v2.0

Legacy supply-mix update (same as `gen_v1.ipynb`) plus the legacy pooled electricity trade layer, updated via `shock_calc` on the proprietary Electricity Maps trade workbook. Standalone: re-parses and re-derives everything `gen_v1.ipynb` does — the two notebooks don't share in-memory state.

**Requires** the Electricity Maps workbook at `paths['export']/_trades_data/trades_<year>.xlsx` (proprietary ToS, not redistributable — governed in nxbase under `electricity_maps/`, not in this repo).

Set `user` and `year` in the first cell, then run top to bottom.

In [ ]:
import mario
import yaml
import os

with open('paths.yml', 'r') as file:  # open the yml file
    paths = yaml.safe_load(file)

user = 'LR'   # change this to your username
year = 2025   # change this to the year you want to build

paths = paths[user]

import pandas as pd
from support.ember_remapping import map_ember_to_classification
import warnings
warnings.filterwarnings("ignore")

Parse, aggregate, and apply the EMBER supply mix — identical to `gen_v1.ipynb`.

In [ ]:
db = mario.parse_from_txt(paths['raw'], table='SUT', mode='flows')

In [ ]:
db.aggregate("support/aggregate_ee.xlsx", ignore_nan=True)

In [ ]:
ee_mix = map_ember_to_classification(
    path = paths['ember'],
    classification = 'EXIO3',
    year = None,
    mode = 'mix',
)

In [ ]:
z = db.z
s = db.s

for region in db.get_index('Region'):
    region_latest_year = ee_mix.loc[(region, slice(None), slice(None))].index.get_level_values(0).max()
    mix_year = year if year <= region_latest_year else region_latest_year

    new_mix = ee_mix.loc[(region, mix_year, slice(None)), 'Value'].to_frame().sort_index(axis=0)
    new_mix.index = new_mix.index.get_level_values(2)

    old_market_share = s.loc[(region, 'Activity', new_mix.index), (region, 'Commodity', 'Electricity')].sum().sum()

    s.loc[(region, 'Activity', new_mix.index), (region, 'Commodity', 'Electricity')] = new_mix['Value'].values * old_market_share

z.update(s)

db.update_scenarios('baseline', z=z)
db.reset_to_coefficients('baseline')

---
## v2.0 — legacy electricity trade

Define the traded commodities and add the legacy pass-through sectors.

In [ ]:
traded_commodities = ['Electricity']

for commodity in traded_commodities:
    new_activities = [f"{commodity} supply"]
    new_commdodities = [f"{commodity} need"]

db.add_sectors(
    new_sectors = new_activities,
    regions = db.get_index("Region"),
    io ="support/add_sectors_activities.xlsx",
    item = "Activity",
    inplace = True,
)

db.add_sectors(
    new_sectors = new_commdodities,
    regions = db.get_index("Region"),
    io ="support/add_sectors_commodities.xlsx",
    item = "Commodity",
    inplace = True,
)

**Demand-side shock**: activities supplying the new commodities consume only the domestic original commodity; consumption of the original commodity moves to the domestic `need` commodity, both for intermediate use and final demand.

In [ ]:
u_new = db.u.copy()
Y_new = db.Y.copy()

U = db.U.copy().loc[(slice(None), "Commodity", traded_commodities), :].T.groupby(level=0).sum().T
Y = Y_new.loc[(slice(None), "Commodity", traded_commodities), :].T.groupby(level=0).sum().T
UY = U + Y

z_new = db.z.copy()

trades_df = {}

for commodity in traded_commodities:
    trades_df[commodity] = pd.DataFrame()
    u_new.loc[:, (slice(None), "Activity", f"{commodity} supply")] *= 0
    oth_activities = [i for i in db.get_index("Activity") if i != f"{commodity} supply"]

    for region in db.get_index("Region"):
        u_new.loc[(region, "Commodity", commodity), (region, "Activity", f"{commodity} supply")] = 1

        ee_consumption_u = db.u.loc[(slice(None), "Commodity", commodity), (region, "Activity", oth_activities)].sum(0).to_frame().T
        ee_consumption_u.index = pd.MultiIndex.from_arrays([[region], ["Commodity"], [f"{commodity} need"]], names=db.u.index.names)

        ee_consumption_Y = db.Y.loc[(slice(None), "Commodity", commodity), (region, "Consumption category", slice(None))].sum(0).to_frame().T
        ee_consumption_Y.index = pd.MultiIndex.from_arrays([[region], ["Commodity"], [f"{commodity} need"]], names=db.Y.index.names)

        u_new.update(ee_consumption_u)
        Y_new.update(ee_consumption_Y)

        u_new.loc[(slice(None), "Commodity", commodity), (region, "Activity", oth_activities)] *= 0
        Y_new.loc[(slice(None), "Commodity", commodity), (region, "Consumption category", slice(None))] *= 0

        trades_df[commodity] = pd.concat([
            trades_df[commodity],
            UY.loc[:, region] / UY.loc[:, region].sum()
        ], axis=1
        )

z_new.update(u_new)

In [ ]:
db.update_scenarios(scenario='baseline', z=z_new, Y=Y_new)
db.reset_to_coefficients('baseline')

**Supply-side shock**: apply the legacy shock workbook (Electricity Maps trades, proprietary — governed in nxbase, not in this repo).

In [ ]:
db.shock_calc(
    os.path.join(paths['export'], "_trades_data", f"trades_{year}.xlsx"),
    z=True, scenario='ee_trades', force_rewrite=True,
)

Export v2.0.

In [ ]:
db.to_txt(
    path = os.path.join(paths['export'], "v2.0", str(year)),
    scenario = 'ee_trades',
    )